### ⭐ 1. Introduction & Overview


Your Goal: Your goal is to predict whether a client will subscribe to a bank term deposit.

### 🔹 2. Import Libraries & Set Up


In [32]:
# =============================================================================    
# MACHINE LEARNING LIBRARIES - SIMPLE IMPORTS
# =============================================================================    

# Set environment variable for scipy array API support
import os
os.environ['SCIPY_ARRAY_API'] = '1'

# Core Data Science Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Gradient Boosting Libraries
import xgboost as xgb
import lightgbm as lgb

# Deep Learning
#import torch
#import torch.nn as nn
#import torch.optim as optim
#import torchvision.transforms as transforms

import tensorflow as tf
from tensorflow import keras

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool

# Computer Vision
import cv2

# Scientific Computing & Statistics
import scipy.stats as stats
from scipy import optimize
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose

# Image Processing
from PIL import Image, ImageDraw, ImageFont

# Sampling and Resampling - Try import, use alternatives if failed
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    IMBLEARN_AVAILABLE = True
except ImportError:
    print("imblearn not available, using sklearn class_weight='balanced' instead")
    IMBLEARN_AVAILABLE = False

# Utilities
import sys
import warnings
import datetime
from pathlib import Path
import pickle
import json

# Configuration
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_palette("husl")
warnings.filterwarnings('ignore')
SEED = 42

In [33]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
bank = pd.read_csv('bank-full.csv', delimiter=';')

In [34]:
train.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [35]:
# Enhanced Feature Engineering for Bank Marketing (Non-redundant)
import numpy as np
import pandas as pd

def create_features(df):
    df = df.copy()

    # Original many_no feature (captures risk aversion pattern)
    def many_no(x):
        if x['default']=='no' and x['housing']=='no' and x['loan']=='no':
            return 21
        if x['default']=='no' and x['housing']=='no'\
        or x['default']=='no' and x['loan']=='no'\
        or x['housing']=='no' and x['loan']=='no':
            return 7
        if x['default']=='no' or x['housing']=='no' or x['loan']=='no':
            return 3
        return 0

    df['many_no'] = df.apply(lambda x: many_no(x), axis=1)

    df['balance_duration'] = df['balance'] * df['duration']  # Financial capacity × engagement
    df['campaign_previous'] = df['campaign'] * df['previous']  # Current effort × past history
    df['age_balance'] = df['age'] * df['balance']  # Life stage × wealth

    df['contact_success_ratio'] = df['previous'] / (df['campaign'] + 1)
    df['days_since_contact'] = np.where(df['pdays'] == -1, 999, df['pdays'])

    df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 50, 65, 100],
                            labels=[0, 1, 2, 3, 4]).astype(int)
    df['balance_category'] = pd.cut(df['balance'], bins=[-np.inf, 0, 1000, 5000, np.inf],
                                    labels=[0, 1, 2, 3]).astype(int)

    return df

bank = create_features(bank)
train = create_features(train)
test = create_features(test)

In [36]:
# Prepare feature matrix (X) and target vector (y) for training
X = train.drop(["y", "id"], axis=1)
y = train["y"]

# Prepare feature matrix (X_bank) and target vector (y_bank) for training
X_bank = bank.drop(["y"], axis=1)
y_bank = bank["y"].map({'yes': 1, 'no': 0})

# Prepare feature matrix (X_test) for testing
X_test = test.drop(["id"], axis=1)

In [37]:
object_cols = train.select_dtypes(include="object").columns

from sklearn.preprocessing import LabelEncoder

for col_name in object_cols:
    le = LabelEncoder()
    X[col_name] = le.fit_transform(X[col_name])
    X_test[col_name] = le.transform(X_test[col_name])
    X_bank[col_name] = le.transform(X_bank[col_name])

In [38]:
train.head(3)

,id,age,job,marital,education,default,balance,housing,loan,contact,...,poutcome,y,many_no,balance_duration,campaign_previous,age_balance,contact_success_ratio,days_since_contact,age_group,balance_category
0,0,42,technician,married,secondary,no,7,no,no,cellular,...,unknown,0,21,819,0,294,0.0,999,2,1
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,...,unknown,0,21,95090,0,19532,0.0,999,2,1
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,...,unknown,0,7,66822,0,21672,0.0,999,2,1


In [39]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

# Use 10-fold stratified cross-validation
n_splits = 10
kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
y_probs = np.zeros(len(X_test))
models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Training fold {fold + 1}/{n_splits} >>>")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    X_train = pd.concat([X_train, X_bank])
    y_train = pd.concat([y_train, y_bank])
    
    model = lgb.LGBMClassifier(
        n_estimators=20000,
        learning_rate=0.06,
        num_leaves=100,
        max_depth=10,
        min_child_samples=9,
        subsample=0.8,
        colsample_bytree=0.5,
        reg_alpha=0.78,
        reg_lambda=3.0,
        max_bin=4523,
        random_state=42,
        verbosity=-1
    )
    
    model.fit(
        X_train, 
        y_train, 
        eval_set=[(X_val, y_val)], 
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(period=500)
        ]
    )

    models.append(model)
    
    # Average predictions across all folds
    y_probs += model.predict_proba(X_test)[:, 1] / n_splits

Training fold 1/10 >>>
Training until validation scores don't improve for 100 rounds
[500]	valid_0's binary_logloss: 0.134942
[1000]	valid_0's binary_logloss: 0.131812
[1500]	valid_0's binary_logloss: 0.130558
[2000]	valid_0's binary_logloss: 0.12992
[2500]	valid_0's binary_logloss: 0.129507
Early stopping, best iteration is:
[2605]	valid_0's binary_logloss: 0.129407
Training fold 2/10 >>>
Training until validation scores don't improve for 100 rounds
[500]	valid_0's binary_logloss: 0.137124
[1000]	valid_0's binary_logloss: 0.133799
[1500]	valid_0's binary_logloss: 0.132358
[2000]	valid_0's binary_logloss: 0.131729
[2500]	valid_0's binary_logloss: 0.13127
Early stopping, best iteration is:
[2558]	valid_0's binary_logloss: 0.131218
Training fold 3/10 >>>
Training until validation scores don't improve for 100 rounds
[500]	valid_0's binary_logloss: 0.138993
[1000]	valid_0's binary_logloss: 0.135548
[1500]	valid_0's binary_logloss: 0.134042
[2000]	valid_0's binary_logloss: 0.133397
Early st

1) [2876]	valid_0's binary_logloss: 0.131302
2) [2804]	valid_0's binary_logloss: 0.132031



In [40]:
from sklearn.metrics import roc_auc_score

best_auc = roc_auc_score(y, model.predict_proba(X)[:, 1])
print(f"Best AUC: {best_auc:.4f}")

Best AUC: 0.9892


1) pre-set features, best AUC: 0.9879
2) new features, best AUC: 0.9892

In [41]:
output = pd.DataFrame({
    'id': test.id,
    'y': y_probs
})

output.to_csv('attempt-lightgbm7.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


Ensemble of LightGBM, XGBoost, CATboost

In [42]:
import optuna, catboost as cb

In [43]:
# XGBoost Hyperparameter Tuning - Complementary to LightGBM/CatBoost
def objective_xgboost(trial):
    # Parameter space for robust regularization (different approach than LightGBM)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 2000, 5000),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.08),
        'max_depth': trial.suggest_int('max_depth', 6, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'subsample': trial.suggest_float('subsample', 0.7, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 5.0),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'random_state': 42,
        'eval_metric': 'logloss',
        'early_stopping_rounds': 100,
        'verbosity': 0
    }

    # 5-fold CV for faster tuning
    cv_scores = []
    kf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for train_idx, val_idx in kf_tune.split(X, y):
        X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
        X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]

        # Add bank data
        X_train_fold = pd.concat([X_train_fold, X_bank])
        y_train_fold = pd.concat([y_train_fold, y_bank])

        model = xgb.XGBClassifier(**params)
        model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], verbose=False)

        pred = model.predict_proba(X_val_fold)[:, 1]
        score = roc_auc_score(y_val_fold, pred)
        cv_scores.append(score)

    return np.mean(cv_scores)

print("Tuning XGBoost parameters...")
study_xgboost = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))       
study_xgboost.optimize(objective_xgboost, n_trials=50)

best_xgboost_params = study_xgboost.best_params
print(f"Best XGBoost CV AUC: {study_xgboost.best_value:.4f}")
print(f"Best XGBoost params: {best_xgboost_params}")

[I 2025-08-07 13:01:26,444] A new study created in memory with name: no-name-d418a1b1-a8db-4ae7-9f4d-ecdf4f262018


Tuning XGBoost parameters...


[I 2025-08-07 13:02:52,437] Trial 0 finished with value: 0.9681654297437244 and parameters: {'n_estimators': 3123, 'learning_rate': 0.0775357153204958, 'max_depth': 9, 'min_child_weight': 5, 'subsample': 0.7312037280884873, 'colsample_bytree': 0.7311989040672405, 'reg_alpha': 0.21035886311957896, 'reg_lambda': 4.46470458309974, 'gamma': 0.6011150117432088}. Best is trial 0 with value: 0.9681654297437244.
[I 2025-08-07 13:05:57,541] Trial 1 finished with value: 0.9685067764131802 and parameters: {'n_estimators': 4124, 'learning_rate': 0.03102922471479012, 'max_depth': 10, 'min_child_weight': 6, 'subsample': 0.7424678221356552, 'colsample_bytree': 0.73636499344142, 'reg_alpha': 0.4484685687215243, 'reg_lambda': 2.216968971838151, 'gamma': 0.5247564316322378}. Best is trial 1 with value: 0.9685067764131802.
[I 2025-08-07 13:08:19,918] Trial 2 finished with value: 0.9684309696188743 and parameters: {'n_estimators': 3296, 'learning_rate': 0.04456145700990209, 'max_depth': 9, 'min_child_weig

Best XGBoost CV AUC: 0.9686
Best XGBoost params: {'n_estimators': 2084, 'learning_rate': 0.04241228443916973, 'max_depth': 8, 'min_child_weight': 4, 'subsample': 0.8894733238422363, 'colsample_bytree': 0.7755272104546164, 'reg_alpha': 1.8455318772526679, 'reg_lambda': 3.297335548087981, 'gamma': 0.48321415961320585}


In [ ]:
# CatBoost Hyperparameter Tuning - Optimized for diversity from LightGBM
def objective_catboost(trial):
    # Parameter space designed to be different from LightGBM
    params = {
        'iterations': trial.suggest_int('iterations', 2000, 5000),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.08),
        'depth': trial.suggest_int('depth', 6, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'border_count': trial.suggest_int('border_count', 128, 255),
        'random_strength': trial.suggest_float('random_strength', 0.5, 2.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_state': 42,
        'verbose': False,
        'early_stopping_rounds': 100
    }

    # 5-fold CV for faster tuning
    cv_scores = []
    kf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for train_idx, val_idx in kf_tune.split(X, y):
        X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
        X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]

        # Add bank data
        X_train_fold = pd.concat([X_train_fold, X_bank])
        y_train_fold = pd.concat([y_train_fold, y_bank])

        model = cb.CatBoostClassifier(**params)
        model.fit(X_train_fold, y_train_fold, eval_set=(X_val_fold, y_val_fold), verbose=False)

        pred = model.predict_proba(X_val_fold)[:, 1]
        score = roc_auc_score(y_val_fold, pred)
        cv_scores.append(score)

    return np.mean(cv_scores)

print("Tuning CatBoost parameters...")
study_catboost = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))      
study_catboost.optimize(objective_catboost, n_trials=50)

best_catboost_params = study_catboost.best_params
print(f"Best CatBoost CV AUC: {study_catboost.best_value:.4f}")
print(f"Best CatBoost params: {best_catboost_params}")

[I 2025-08-07 15:50:39,142] A new study created in memory with name: no-name-31037c8e-fb7c-4f8c-86cc-f3a82f5285aa


Tuning CatBoost parameters...


[I 2025-08-07 15:59:19,874] Trial 0 finished with value: 0.9667201844179077 and parameters: {'iterations': 3123, 'learning_rate': 0.0775357153204958, 'depth': 9, 'l2_leaf_reg': 6.387926357773329, 'border_count': 147, 'random_strength': 0.7339917805043039, 'bagging_temperature': 0.05808361216819946}. Best is trial 0 with value: 0.9667201844179077.
[I 2025-08-07 16:10:40,762] Trial 1 finished with value: 0.9674417394062009 and parameters: {'iterations': 4599, 'learning_rate': 0.060055750587160436, 'depth': 9, 'l2_leaf_reg': 1.185260448662222, 'border_count': 252, 'random_strength': 1.7486639612006325, 'bagging_temperature': 0.21233911067827616}. Best is trial 1 with value: 0.9674417394062009.
[I 2025-08-07 16:20:45,734] Trial 2 finished with value: 0.9664948735192249 and parameters: {'iterations': 2545, 'learning_rate': 0.03917022549267169, 'depth': 7, 'l2_leaf_reg': 5.72280788469014, 'border_count': 183, 'random_strength': 0.9368437102970628, 'bagging_temperature': 0.6118528947223795}. 

In [ ]:
# Train finetuned CatBoost with same CV structure as your LightGBM   
print("Training finetuned CatBoost...")
y_probs_cat_tuned = np.zeros(len(X_test))
catboost_models_tuned = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"CatBoost fold {fold + 1}/{n_splits}")
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]

    # Add bank data like your LightGBM
    X_train_fold = pd.concat([X_train_fold, X_bank])
    y_train_fold = pd.concat([y_train_fold, y_bank])

    model = cb.CatBoostClassifier(**best_catboost_params)
    model.fit(
        X_train_fold,
        y_train_fold,
        eval_set=(X_val_fold, y_val_fold),
        verbose=500,
        use_best_model=True
    )

    catboost_models_tuned.append(model)
    y_probs_cat_tuned += model.predict_proba(X_test)[:, 1] / n_splits

print("CatBoost training completed!")

In [ ]:
# Train finetuned XGBoost with same CV structure as your LightGBM    
print("Training finetuned XGBoost...")
y_probs_xgb_tuned = np.zeros(len(X_test))
xgboost_models_tuned = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"XGBoost fold {fold + 1}/{n_splits}")
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]

    # Add bank data like your LightGBM
    X_train_fold = pd.concat([X_train_fold, X_bank])
    y_train_fold = pd.concat([y_train_fold, y_bank])

    model = xgb.XGBClassifier(**best_xgboost_params)
    model.fit(
        X_train_fold,
        y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=500
    )

    xgboost_models_tuned.append(model)
    y_probs_xgb_tuned += model.predict_proba(X_test)[:, 1] / n_splits

print("XGBoost training completed!")

In [ ]:
# Create ensemble from all 3 finetuned models
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score

print("=== CREATING OPTIMIZED 3-MODEL ENSEMBLE ===")

# Generate out-of-fold predictions for validation
def get_oof_predictions_simple(models_list, X_data, y_data, kf):
    """Generate out-of-fold predictions"""
    oof_preds = np.zeros(len(X_data))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_data, y_data)):
        model = models_list[fold]
        oof_preds[val_idx] = model.predict_proba(X_data.iloc[val_idx])[:, 1]

    return oof_preds

# Get OOF predictions for ensemble optimization
lgb_oof = get_oof_predictions_simple(models, X, y, kf)
cat_oof = get_oof_predictions_simple(catboost_models_tuned, X, y, kf)
xgb_oof = get_oof_predictions_simple(xgboost_models_tuned, X, y, kf)

# Individual model scores
lgb_score = roc_auc_score(y, lgb_oof)
cat_score = roc_auc_score(y, cat_oof)
xgb_score = roc_auc_score(y, xgb_oof)

print(f"Individual Model CV Scores:")
print(f"LightGBM:     {lgb_score:.4f}")
print(f"CatBoost:     {cat_score:.4f}")
print(f"XGBoost:      {xgb_score:.4f}")
print()

# Optimize ensemble weights
def ensemble_loss(weights, *args):
    lgb_pred, cat_pred, xgb_pred, y_true = args
    weights = weights / weights.sum()  # Normalize
    ensemble_pred = weights[0] * lgb_pred + weights[1] * cat_pred + weights[2] * xgb_pred
    return -roc_auc_score(y_true, ensemble_pred)  # Negative because we minimize

result = minimize(
    ensemble_loss,
    x0=[1, 1, 1],  # Initial equal weights
    args=(lgb_oof, cat_oof, xgb_oof, y),
    bounds=[(0.01, 5), (0.01, 5), (0.01, 5)],
    method='L-BFGS-B'
)

optimal_weights = result.x / result.x.sum()
print(f"Optimal weights:")
print(f"  LightGBM: {optimal_weights[0]:.3f}")
print(f"  CatBoost: {optimal_weights[1]:.3f}")
print(f"  XGBoost:  {optimal_weights[2]:.3f}")

# Create final ensemble predictions
ensemble_oof = (optimal_weights[0] * lgb_oof +
                optimal_weights[1] * cat_oof +
                optimal_weights[2] * xgb_oof)

ensemble_test = (optimal_weights[0] * y_probs +
                optimal_weights[1] * y_probs_cat_tuned +
                optimal_weights[2] * y_probs_xgb_tuned)

ensemble_score = roc_auc_score(y, ensemble_oof)
improvement = ensemble_score - lgb_score

print(f"\nEnsemble Performance:")
print(f"Ensemble CV AUC: {ensemble_score:.4f}")
print(f"Improvement:     {improvement:+.4f} over best single model")

if improvement > 0:
    print("✅ Ensemble improves performance!")
else:
    print("⚠️  Ensemble doesn't improve - consider using LightGBM only")

In [ ]:
# Save final ensemble results
if improvement > 0:
    # Use ensemble if it improves
    final_predictions = ensemble_test
    method_used = "Weighted Ensemble"
    final_score = ensemble_score
else:
    # Use LightGBM if ensemble doesn't help
    final_predictions = y_probs
    method_used = "LightGBM Only"
    final_score = lgb_score

# Save main submission
final_submission = pd.DataFrame({
    'id': test.id,
    'y': final_predictions
})

final_submission.to_csv('final_3model_ensemble.csv', index=False)
print(f"Final submission saved as 'final_3model_ensemble.csv'")
print(f"Method used: {method_used}")
print(f"CV AUC: {final_score:.4f}")

# Also save all predictions for comparison
all_predictions = pd.DataFrame({
    'id': test.id,
    'lightgbm': y_probs,
    'catboost_tuned': y_probs_cat_tuned,
    'xgboost_tuned': y_probs_xgb_tuned,
    'weighted_ensemble': ensemble_test
})

all_predictions.to_csv('all_model_predictions.csv', index=False)
print("All model predictions saved as 'all_model_predictions.csv'")

print(f"\n🏆 FINAL RESULTS:")
print(f"{'='*50}")
print(f"Best method: {method_used}")
print(f"Final CV AUC: {final_score:.4f}")
if improvement > 0:
    print(f"Improvement: +{improvement:.4f} over LightGBM")
else:
    print("Ensemble didn't improve - using single model")
print(f"{'='*50}")